In [1]:
TARGET = "st_xfer_time_ms"

NUMERIC_COLS = [
    "st_files",
    "st_dirs",
    "st_successful",
    "st_failed",
    "st_expired",
    "st_canceled",
    "st_bytes_xfered",
    "st_faults",
    "st_files_skipped",
    "st_skipped_errors",
    "st_xfer_time_ms"
]

CATEGORICAL_COLS = [
    "grp_status",
    "src_host_ep_id",
    "dst_host_ep_id",
    "encrypt_data",
    "grp_delete",
]

DROP_COLS = ["grp_uuid", "user_id", "request_time", "complete_time"]

In [2]:
import numpy as np
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr

def pinball_loss(y_true, y_pred, tau):
    diff = y_true - y_pred
    return np.mean(np.maximum(tau * diff, (tau - 1) * diff))


def evaluate_model(model, X_tr, y_tr, X_te, y_te, y_te_raw):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)

    return {
        "mae_log": mean_absolute_error(y_te, pred),
        "spearman": spearmanr(y_te, pred).correlation,
        "mae_log_top90": mean_absolute_error(
            y_te[y_te_raw >= np.percentile(y_te_raw, 90)],
            pred[y_te_raw >= np.percentile(y_te_raw, 90)]
        ),
        "mae_log_top99": mean_absolute_error(
            y_te[y_te_raw >= np.percentile(y_te_raw, 99)],
            pred[y_te_raw >= np.percentile(y_te_raw, 99)]
        ),
        "pinball_90": pinball_loss(y_te, pred, 0.9),
        "pinball_95": pinball_loss(y_te, pred, 0.95),
    }


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("num", "passthrough", NUMERIC_COLS),
            ("cat", OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ), CATEGORICAL_COLS),
        ]
    )



In [4]:
from xgboost import XGBRegressor

def make_xgb():
    return XGBRegressor(
        n_estimators=500,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        tree_method="hist",       # was "hist"
        predictor="gpu_predictor",    # keep inference on GPU (optional but recommended)
        device="cuda",                # xgboost>=2.0, or use `gpu_id=0` on older releases
        random_state=0,
        n_jobs=0,                     # GPU path ignores CPU threads
    )


In [5]:
from sklearn.model_selection import train_test_split
from pathlib import Path
import json
import pandas as pd

def downstream_eval_xgb(real_df, synth_df, seed=0):
    real_train, real_test = train_test_split(
        real_df, test_size=0.2, random_state=seed
    )

    # Target
    y_tr = np.log1p(real_train[TARGET].values)
    y_te = np.log1p(real_test[TARGET].values)
    y_te_raw = real_test[TARGET].values
    y_syn = np.log1p(synth_df[TARGET].values)

    # Drop columns
    real_train = real_train.drop(columns=DROP_COLS)
    real_test = real_test.drop(columns=DROP_COLS)
    synth_df = synth_df.drop(columns=DROP_COLS)

    # Preprocess (fit on real-train ONLY)
    pre = make_preprocessor()
    X_tr = pre.fit_transform(real_train)
    X_te = pre.transform(real_test)
    X_syn = pre.transform(synth_df)

    model = make_xgb()

    rr = evaluate_model(model, X_tr, y_tr, X_te, y_te, y_te_raw)
    sr = evaluate_model(model, X_syn, y_syn, X_te, y_te, y_te_raw)

    return {
        "RR": rr,
        "SR": sr,
        "Delta": {k: sr[k] - rr[k] for k in rr},
    }


In [ ]:
real = pd.read_csv("../datasets/filtered.csv", engine="pyarrow")
synth_df = pd.read_csv("../output/test.csv", engine="pyarrow")

column = "st_xfer_time_ms"
k = 3  # number of sigmas

# Compute statistics
mu = real[column].mean()
sigma = real[column].std()

# Filter DataFrame
real_filtered = real[(real[column] >= mu - k * sigma) &
                 (real
                [column] <= mu + k * sigma)]
synth_filtered = synth_df[(synth_df[column] >= mu - k * sigma) &
                 (synth_df
                [column] <= mu + k * sigma)]

results = downstream_eval_xgb(real_filtered, synth_filtered)

output_path = Path("../output/downstream-results3.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
json_payload = json.dumps(results, indent=2, default=float)
output_path.write_text(json_payload)
print(json_payload)
print(f"Saved downstream metrics to {output_path}")


/home/seongho/gmm-synth-transfer/.venv/lib/python3.13/site-packages/xgboost/core.py:158: UserWarning: [23:22:12] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)
/home/seongho/gmm-synth-transfer/.venv/lib/python3.13/site-packages/xgboost/core.py:158: UserWarning: [23:26:55] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  warnings.warn(smsg, UserWarning)
/home/seongho/gmm-synth-transfer/.venv/lib/python3.13/site-packages/xgboost/core.py:158: UserWarning: [23:27:19] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "predictor" } ar

{
  "RR": {
    "mae_log": 0.016338365766785785,
    "spearman": 0.9999036022822535,
    "mae_log_top90": 0.056523607792039084,
    "mae_log_top99": 0.16059070710332635,
    "pinball_90": 0.00818377043973115,
    "pinball_95": 0.008185593884273432
  },
  "SR": {
    "mae_log": 0.022501166036549888,
    "spearman": 0.9998258184429889,
    "mae_log_top90": 0.062453633848441215,
    "mae_log_top99": 0.18915706275594824,
    "pinball_90": 0.009331068452269298,
    "pinball_95": 0.009091129131518593
  },
  "Delta": {
    "mae_log": 0.006162800269764103,
    "spearman": -7.77838392646002e-05,
    "mae_log_top90": 0.005930026056402131,
    "mae_log_top99": 0.02856635565262189,
    "pinball_90": 0.0011472980125381473,
    "pinball_95": 0.0009055352472451608
  }
}
Saved downstream metrics to output/downstream-results3.json
